# Knowledge Graph-Based JD-CV Matching & Ranking
This notebook implements a **Knowledge Graph (KG)** approach to parse resumes and a Job Description, represent their entities and relationships, and score candidates based on structural graph coverage rather than simple keyword lists.

In [ ]:
# 1. Install Ollama (zstd is required by the installer but missing from the base Colab image)
!apt-get -qq update && apt-get -qq install -y zstd
!apt-get update && apt-get install -y pciutils
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
# 2. Start local Ollama server in background
import os, subprocess, time

env = os.environ.copy()
env["OLLAMA_ORIGINS"] = "*"
env["OLLAMA_HOST"] = "0.0.0.0"

# Kill any process already occupying port 11434
subprocess.run(["bash", "-c", "fuser -k 11434/tcp || true"])
time.sleep(3)

ollama_process = subprocess.Popen(["ollama", "serve"], env=env)
time.sleep(5)
print("Ollama server started (pid:", ollama_process.pid, ")")

# Run a local tags check to verify it is running
status = subprocess.run(
    ["curl", "-s", "-o", "/dev/null", "-w", "%{http_code}", "http://localhost:11434/api/tags"],
    capture_output=True, text=True,
).stdout
print("Local API check status (expecting 200):", status)

In [ ]:
# 3. Pull a small and fast model for processing (Gemma 2B)
!ollama pull gemma:2b

In [ ]:
# 4. Load the Candidate CVs and define the JD
import pandas as pd
import os

csv_path = "extracted_cvs_30.csv"
if not os.path.exists(csv_path):
    if os.path.exists("resources/Tester/extracted_cvs_30.csv"):
        csv_path = "resources/Tester/extracted_cvs_30.csv"
    elif os.path.exists("/content/extracted_cvs_30.csv"):
        csv_path = "/content/extracted_cvs_30.csv"
    else:
        print("Warning: extracted_cvs_30.csv not found locally. Please verify path or upload files.")

if os.path.exists(csv_path):
    df_cvs = pd.read_csv(csv_path)
    print(f"Loaded {len(df_cvs)} candidates from {csv_path}")
else:
    # Fallback mock data in case notebook is run standalone without environment
    df_cvs = pd.DataFrame([
        {"filename": "1.docx", "raw_text": "Java developer with 3 years of C# experience in Web applications"},
        {"filename": "2.docx", "raw_text": "Expert in C#, ASP.NET, and SQL Server. 6 years of Microsoft Stack experience."}
    ])
    print("Using fallback mock dataframe because CSV was not found.")

# Define the .NET Job Description
dotnet_jd = """
Primary Purpose: Development and maintenance of software applications built using Microsoft stack.
Minimum Skills and Competencies:
5+ years of experience in Microsoft technology stack.
Expert in C#, JavaScript, MSSQL 2012 or above.
Proficient with MVC, Angular, Asp.Net, JQuery.
Hands on experience with Visual Studio & TFS as source control tool.
Must have good understanding of SOLID object oriented programming (OOP) concepts.
Must have experience with WCF.
Desired Skills:
Relevant Microsoft certification.
Experience with Web API, Entity Framework.
"""
print("JD Text loaded successfully.")

In [ ]:
# 5. Setup LLM Triple/Graph Extraction Helpers
import json
import requests
import time

MODEL_NAME = "gemma:2b"

SYSTEM_PROMPT = """
You are an AI pipeline that extracts structured Knowledge Graphs from text.
Extract critical entities (SKILL, ROLE, COMPANY, DEGREE) and their relationships.

You MUST output strictly valid JSON matching this schema, without any markdown delimiters, explanation, or code fences:
{
  "nodes": [
    {"id": "entity_name", "type": "SKILL|ROLE|COMPANY|DEGREE"}
  ],
  "edges": [
    {"source": "entity_name", "target": "entity_name", "relation": "HAS_SKILL|REQUIRES_SKILL|AT_COMPANY|HELD_ROLE", "duration_years": 0}
  ]
}

Standardize entity identifiers (e.g. use 'C#', 'JavaScript', 'SQL Server' consistently). Keep entity IDs concise.
"""

def extract_graph_from_text(text):
    url = "http://localhost:11434/api/chat"
    # Keep text length within model context limits
    trimmed_text = text[:8000]
    
    payload = {
        "model": MODEL_NAME,
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": trimmed_text}
        ],
        "stream": False,
        "format": "json",
        "options": {
            "temperature": 0.0
        }
    }
    
    try:
        res = requests.post(url, json=payload, timeout=60)
        res.raise_for_status()
        content = res.json()["message"]["content"].strip()
        return json.loads(content)
    except Exception as e:
        # Print debug and return empty graphs on failure
        print(f"Extraction error: {e}")
        return {"nodes": [], "edges": []}
print("Extraction helper function initialized.")

In [ ]:
# 6. Perform Graph Extraction for JD and CVs
print("Extracting JD Graph...")
jd_graph_data = extract_graph_from_text(dotnet_jd)
print(f"JD Nodes: {len(jd_graph_data.get('nodes', []))}, Edges: {len(jd_graph_data.get('edges', []))}")

print("Extracting Candidate CV Graphs (this may take a few minutes)...")
cv_graphs = {}
for idx, row in df_cvs.iterrows():
    fn = row["filename"]
    txt = row["raw_text"]
    print(f"Processing ({idx+1}/{len(df_cvs)}): {fn}...")
    cv_graphs[fn] = extract_graph_from_text(txt)

In [ ]:
# 7. Convert extracted datasets to networkx structures
import networkx as nx

def build_nx_graph(data):
    G = nx.DiGraph()
    for node in data.get("nodes", []):
        G.add_node(node["id"], type=node.get("type", "UNKNOWN"))
    for edge in data.get("edges", []):
        G.add_edge(
            edge["source"], 
            edge["target"], 
            relation=edge.get("relation", "UNKNOWN"),
            duration_years=float(edge.get("duration_years") or 0.0)
        )
    return G

G_jd = build_nx_graph(jd_graph_data)
cv_nx_graphs = {fn: build_nx_graph(data) for fn, data in cv_graphs.items()}
print("NetworkX Graph representations constructed.")

In [ ]:
# 8. Define Graph Matching, Scoring and Ranking Engine
def evaluate_matching(G_jd, G_cv):
    # 1. Identify skills required in the Job Description
    jd_skills = {n.lower() for n, attr in G_jd.nodes(data=True) if attr.get("type") == "SKILL"}
    if not jd_skills:
        # Default fallback standard required stack if extraction was blank
        jd_skills = {"c#", "javascript", "mssql", "mvc", "angular", "asp.net", "jquery", "wcf"}
        
    # 2. Check candidate CV graph nodes
    cv_nodes = {n.lower() for n in G_cv.nodes()}
    
    # Compute skill intersection
    matched_skills = []
    for s in jd_skills:
        if s in cv_nodes:
            matched_skills.append(s)
        else:
            # Check substring matches to handle synonym variants
            for cv_n in cv_nodes:
                if s in cv_n or cv_n in s:
                    matched_skills.append(s)
                    break
                    
    skill_score = len(matched_skills) / len(jd_skills) * 100 if jd_skills else 0.0
    
    # 3. Constraint checking: Verify if candidate has 5+ years in Microsoft stack
    duration_years = 0.0
    for u, v, d in G_cv.edges(data=True):
        edge_text = f"{u} {v} {d.get('relation', '')}".lower()
        if any(term in edge_text for term in ["c#", ".net", "microsoft"]):
            duration_years = max(duration_years, d.get("duration_years", 0.0))
            
    experience_qualified = duration_years >= 5.0
    
    # Final weight: 70% skill coverage + 30% experience constraint match
    experience_score = 30.0 if experience_qualified else (duration_years / 5.0 * 30.0)
    final_score = (skill_score * 0.7) + experience_score
    
    justification = f"Matched Skills: {matched_skills}. Experience: {duration_years} yrs in Microsoft stack."
    return round(final_score, 2), justification

results = []
for fn, G_cv in cv_nx_graphs.items():
    score, justification = evaluate_matching(G_jd, G_cv)
    results.append({
        "filename": fn,
        "score": score,
        "justification": justification
    })

df_rankings = pd.DataFrame(results).sort_values(by="score", ascending=False).reset_index(drop=True)
df_rankings["rank"] = df_rankings.index + 1
print("Ranking calculations finished.")

In [ ]:
# 9. Display Ranking Output and Visualize Graph
print("=== CANDIDATE RANKINGS (KNOWLEDGE GRAPH MATCH) ===")
print(df_rankings[["rank", "filename", "score", "justification"]].to_string(index=False))

import matplotlib.pyplot as plt
try:
    plt.figure(figsize=(10, 6))
    pos = nx.spring_layout(G_jd)
    nx.draw(G_jd, pos, with_labels=True, node_color='lightgreen', node_size=1500, font_size=8, arrowsize=10)
    plt.title("Job Description Knowledge Graph")
    plt.show()
except Exception as e:
    print("Visualization skipped:", e)